# Rolling quadrimesters with new definitions

This notebook rebuilds the rolling four-month pipeline with the revised variables.

## Setup

We define paths, constants, and reusable helpers.

In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
import re
import unicodedata

import numpy as np
import pandas as pd


OUTPUT_DIR = Path.cwd()
DATA_DIR = OUTPUT_DIR.parents[3]

COMMENTS_SOURCE = DATA_DIR / "ig_comments_clean.csv"
POSTS_SOURCE = DATA_DIR / "ig_posts_clean.csv"

COMMENTS_OUTPUT = OUTPUT_DIR / "ig_comments_sep2022_start.csv"
POSTS_OUTPUT = OUTPUT_DIR / "ig_posts_sep2022_start.csv"
METRICS_OUTPUT = OUTPUT_DIR / "ig_user_rolling_quadrimesters_new_definitions.csv"
CLASSIFICATION_OUTPUT = OUTPUT_DIR / "Ig_RCEDTG_classification.csv"
USER_PROFILE_OUTPUT = OUTPUT_DIR / "Ig_RCEDTG_user_frequency_profiles.csv"
KMEANS_K4_ASSIGNMENT_OUTPUT = OUTPUT_DIR / "Ig_RCEDTG_kmeans_k4_user_clusters.csv"
KMEANS_K4_SUMMARY_OUTPUT = OUTPUT_DIR / "Ig_RCEDTG_kmeans_k4_cluster_summary.csv"
MONTHLY_CLUSTER_MATRIX_OUTPUT = OUTPUT_DIR / "Ig_RCEDTG_k4_monthly_cluster_matrix.csv"
TRANSITION_MATRIX_DIR = OUTPUT_DIR / "Transition matrixes 4m"
SUMMARY_TABLES_DIR = OUTPUT_DIR / "Summary tables"
KMEANS_K3_PLOT = OUTPUT_DIR / "Ig_RCEDTG_kmeans_clusters_k3.png"
KMEANS_K4_PLOT = OUTPUT_DIR / "Ig_RCEDTG_kmeans_clusters_k4.png"
STATE_PERSISTENCE_SUMMARY_PNG = SUMMARY_TABLES_DIR / "ig_state_persistence_summary_4clusters.png"
STATE_EXIT_DESTINATIONS_PNG = SUMMARY_TABLES_DIR / "ig_state_exit_destinations_top_4clusters.png"
STATE_EXIT_RISK_SUMMARY_PNG = SUMMARY_TABLES_DIR / "ig_state_exit_risk_summary_4clusters.png"
SUNKEY_DIAGRAM_DIR = OUTPUT_DIR / "Sunkey diagram RCEDTG"
SUNKEY_DIAGRAM_PNG = SUNKEY_DIAGRAM_DIR / "sankey_2023_2026_jan_may_sep_checkpoint_flows_rcedtg.png"
WINDOWLEVEL_K4_SUMMARY_OUTPUT = OUTPUT_DIR / "Ig_RCEDTG_windowlevel_kmeans_k4_cluster_summary.csv"
WINDOWLEVEL_K4_MONTHLY_MATRIX_OUTPUT = OUTPUT_DIR / "Ig_RCEDTG_windowlevel_kmeans_k4_monthly_cluster_matrix.csv"

START_MONTH = pd.Timestamp("2022-09-01")
WINDOW_MONTHS = 4
ANALYSIS_END_EXCLUSIVE = pd.Timestamp("2026-04-01")
LAST_WINDOW_START = ANALYSIS_END_EXCLUSIVE - pd.DateOffset(months=WINDOW_MONTHS)
CREATOR_USERNAME = "camihawke"
MIDNIGHT_BACKOFF = pd.Timedelta(minutes=2)

COMMENT_DTYPES = {
    "comment_id": "string",
    "media_id": "string",
    "text": "string",
    "timestamp": "string",
    "from_id": "string",
    "from_username": "string",
    "parent_id": "string",
    "is_reply": "boolean",
    "is_creator": "boolean",
}

POST_DTYPES = {
    "media_id": "string",
    "timestamp": "string",
}

NON_EMOJI_TOKEN_RE = re.compile(
    r"https?://\S+|[#@]?\w+(?:[.'_-]\w+)*",
    flags=re.UNICODE,
)

EMOJI_RANGES = (
    (0x00A9, 0x00A9),
    (0x00AE, 0x00AE),
    (0x203C, 0x203C),
    (0x2049, 0x2049),
    (0x2122, 0x2122),
    (0x2139, 0x2139),
    (0x2194, 0x2199),
    (0x21A9, 0x21AA),
    (0x231A, 0x231B),
    (0x2328, 0x2328),
    (0x23CF, 0x23CF),
    (0x23E9, 0x23F3),
    (0x23F8, 0x23FA),
    (0x24C2, 0x24C2),
    (0x25AA, 0x25AB),
    (0x25B6, 0x25B6),
    (0x25C0, 0x25C0),
    (0x25FB, 0x25FE),
    (0x2600, 0x27BF),
    (0x2934, 0x2935),
    (0x2B05, 0x2B07),
    (0x2B1B, 0x2B1C),
    (0x2B50, 0x2B50),
    (0x2B55, 0x2B55),
    (0x3030, 0x3030),
    (0x303D, 0x303D),
    (0x3297, 0x3297),
    (0x3299, 0x3299),
    (0x1F000, 0x1FAFF),
)

EMOJI_CONNECTORS = {0x200D, 0x20E3, 0xFE0E, 0xFE0F}
KEYCAP_BASE_CHARS = set("0123456789#*")


@dataclass(frozen=True)
class WindowSpec:
    window_id: str
    window_label: str
    window_index: int
    start: pd.Timestamp
    end: pd.Timestamp

    @property
    def end_exclusive(self) -> pd.Timestamp:
        return self.end + pd.Timedelta(days=1)

    @property
    def total_days(self) -> int:
        return int((self.end - self.start).days + 1)


def build_user_key(comments: pd.DataFrame) -> pd.Series:
    from_username = comments["from_username"].fillna("").astype("string").str.strip()
    from_username_norm = from_username.str.casefold()
    from_id = comments["from_id"].fillna("").astype("string").str.strip()
    from_id = from_id.mask(from_id == "")
    return from_id.fillna("username:" + from_username_norm)


def is_emojiish_char(char: str) -> bool:
    codepoint = ord(char)
    if codepoint in EMOJI_CONNECTORS or 0x1F3FB <= codepoint <= 0x1F3FF:
        return True
    for start, end in EMOJI_RANGES:
        if start <= codepoint <= end:
            return True
    return False


def is_regional_indicator(codepoint: int) -> bool:
    return 0x1F1E6 <= codepoint <= 0x1F1FF


def is_skin_tone_modifier(codepoint: int) -> bool:
    return 0x1F3FB <= codepoint <= 0x1F3FF


def is_emoji_base_char(char: str) -> bool:
    codepoint = ord(char)
    if codepoint in EMOJI_CONNECTORS or is_skin_tone_modifier(codepoint):
        return False
    if unicodedata.combining(char) > 0:
        return False
    return is_emojiish_char(char)


def is_combining_or_extender(char: str) -> bool:
    codepoint = ord(char)
    return (
        unicodedata.combining(char) > 0
        or codepoint in {0x20E3, 0xFE0E, 0xFE0F}
        or is_skin_tone_modifier(codepoint)
    )


def consume_keycap_sequence(value: str, start: int) -> int | None:
    if value[start] not in KEYCAP_BASE_CHARS:
        return None

    end = start + 1
    if end < len(value) and ord(value[end]) == 0xFE0F:
        end += 1

    if end < len(value) and ord(value[end]) == 0x20E3:
        return end + 1

    return None


def consume_emoji_cluster(value: str, start: int) -> tuple[str, int] | None:
    keycap_end = consume_keycap_sequence(value, start)
    if keycap_end is not None:
        return value[start:keycap_end], keycap_end

    if start >= len(value):
        return None

    char = value[start]
    codepoint = ord(char)
    if is_regional_indicator(codepoint):
        end = start + 1
        if end < len(value) and is_regional_indicator(ord(value[end])):
            end += 1
        return value[start:end], end

    if not is_emoji_base_char(char):
        return None

    end = start + 1
    while end < len(value) and is_combining_or_extender(value[end]):
        end += 1

    while end < len(value) and ord(value[end]) == 0x200D:
        end += 1
        if end >= len(value):
            break
        end += 1
        while end < len(value) and is_combining_or_extender(value[end]):
            end += 1

    return value[start:end], end


def count_words_with_emoji(text: object) -> int:
    if pd.isna(text):
        return 0

    value = str(text).strip()
    if not value:
        return 0

    count = 0
    last_emoji_cluster: str | None = None
    non_emoji_chunk: list[str] = []

    def flush_non_emoji() -> int:
        nonlocal non_emoji_chunk
        if not non_emoji_chunk:
            return 0
        piece = "".join(non_emoji_chunk)
        non_emoji_chunk = []
        return len(NON_EMOJI_TOKEN_RE.findall(piece))

    index = 0
    while index < len(value):
        char = value[index]

        if char.isspace():
            count += flush_non_emoji()
            last_emoji_cluster = None
            index += 1
            continue

        emoji_cluster = consume_emoji_cluster(value, index)
        if emoji_cluster is not None:
            count += flush_non_emoji()
            cluster_text, next_index = emoji_cluster
            if cluster_text != last_emoji_cluster:
                count += 1
                last_emoji_cluster = cluster_text
            index = next_index
            continue

        last_emoji_cluster = None
        non_emoji_chunk.append(char)
        index += 1

    count += flush_non_emoji()
    return count


def build_window_specs() -> list[WindowSpec]:
    windows: list[WindowSpec] = []
    current_start = START_MONTH
    index = 1

    while current_start <= LAST_WINDOW_START:
        current_end = current_start + pd.DateOffset(months=WINDOW_MONTHS) - pd.Timedelta(days=1)
        windows.append(
            WindowSpec(
                window_id=f"w{index:02d}",
                window_label=f"{current_start.strftime('%Y-%m')} to {current_end.strftime('%Y-%m')}",
                window_index=index,
                start=current_start,
                end=current_end,
            )
        )
        current_start += pd.DateOffset(months=1)
        index += 1

    return windows


## Build base inputs

We keep top-level non-creator comments from September 2022 onward only when they refer to posts published from September 2022 onward, remove zero-word comments and users with one comment, and keep matching posts.

In [ ]:
def load_all_comments() -> pd.DataFrame:
    comments = pd.read_csv(COMMENTS_SOURCE, dtype=COMMENT_DTYPES, low_memory=False)
    comments["timestamp"] = pd.to_datetime(comments["timestamp"], errors="coerce")
    comments = comments.dropna(subset=["comment_id", "media_id", "timestamp"]).copy()
    comments["from_username"] = comments["from_username"].fillna("").astype("string").str.strip()
    comments["from_id"] = comments["from_id"].fillna("").astype("string").str.strip()
    comments["parent_id"] = comments["parent_id"].fillna("").astype("string").str.strip()
    comments["user_key"] = build_user_key(comments)
    return comments


def load_all_posts() -> pd.DataFrame:
    posts = pd.read_csv(POSTS_SOURCE, dtype=POST_DTYPES, low_memory=False)
    posts["timestamp"] = pd.to_datetime(posts["timestamp"], errors="coerce")
    posts = posts.dropna(subset=["media_id", "timestamp"]).copy()
    return posts


all_comments = load_all_comments()
all_posts = load_all_posts()
eligible_posts = all_posts.loc[
    (all_posts["timestamp"] >= START_MONTH) & (all_posts["timestamp"] < ANALYSIS_END_EXCLUSIVE)
].copy()
eligible_media_ids = pd.Index(eligible_posts["media_id"].drop_duplicates())

creator_mask_all = (all_comments["is_creator"] == True) | (
    all_comments["from_username"].str.casefold() == CREATOR_USERNAME
)
history_comments = all_comments.loc[~creator_mask_all].copy()
history_comments["word_count"] = history_comments["text"].map(count_words_with_emoji).astype(int)
history_comments = history_comments.loc[history_comments["word_count"] > 0].copy()

base_comments = history_comments.loc[
    (history_comments["timestamp"] >= START_MONTH) & (history_comments["timestamp"] < ANALYSIS_END_EXCLUSIVE)
].copy()
base_comments = base_comments.loc[base_comments["media_id"].isin(eligible_media_ids)].copy()

reply_mask = (base_comments["is_reply"] == True) | base_comments["parent_id"].ne("")
base_comments = base_comments.loc[~reply_mask].copy()

user_comment_counts = base_comments.groupby("user_key").size()
keep_user_keys = user_comment_counts[user_comment_counts > 1].index

base_comments = base_comments.loc[base_comments["user_key"].isin(keep_user_keys)].copy()
history_comments = history_comments.loc[history_comments["user_key"].isin(keep_user_keys)].copy()

base_comments = base_comments.sort_values(["timestamp", "media_id", "comment_id"], kind="mergesort").reset_index(drop=True)
history_comments = history_comments.sort_values(["user_key", "timestamp", "comment_id"], kind="mergesort").reset_index(drop=True)

base_posts = eligible_posts.loc[eligible_posts["media_id"].isin(base_comments["media_id"].drop_duplicates())].copy()
base_posts = base_posts.sort_values(["timestamp", "media_id"], kind="mergesort").reset_index(drop=True)

comments_output = base_comments.drop(columns=["word_count"])
comments_output.to_csv(COMMENTS_OUTPUT, index=False, encoding="utf-8-sig")
base_posts.to_csv(POSTS_OUTPUT, index=False, encoding="utf-8-sig")

print(f"Saved {len(comments_output):,} rows to {COMMENTS_OUTPUT.name}")
print(f"Saved {len(base_posts):,} rows to {POSTS_OUTPUT.name}")
print(f"Users retained: {base_comments['user_key'].nunique():,}")
print(f"Posts retained: {base_posts['media_id'].nunique():,}")


## Compute rolling metrics

We calculate Recency, Coverage, Engagement, Tenure, and Gini for each user and rolling quadrimester.

In [ ]:
TARGET_COLUMNS = [
    "window_id",
    "window_label",
    "window_index",
    "window_start",
    "window_end",
    "window_total_days",
    "posts_in_window",
    "user_key",
    "user_id",
    "username",
    "comment_count",
    "last_comment_timestamp",
    "recency",
    "coverage",
    "engagement",
    "delay",
    "tenure",
    "gini",
]


def build_post_delay_reference(posts: pd.DataFrame, comments: pd.DataFrame) -> pd.DataFrame:
    first_comment_ts = (
        comments.groupby("media_id", sort=False)["timestamp"]
        .min()
        .rename("first_comment_timestamp")
        .reset_index()
    )
    post_reference = posts.merge(first_comment_ts, on="media_id", how="left")

    midnight_mask = (
        (post_reference["timestamp"].dt.hour == 0)
        & (post_reference["timestamp"].dt.minute == 0)
        & (post_reference["timestamp"].dt.second == 0)
    )
    can_backfill_mask = midnight_mask & post_reference["first_comment_timestamp"].notna()
    post_reference["delay_post_timestamp"] = post_reference["timestamp"]
    post_reference.loc[can_backfill_mask, "delay_post_timestamp"] = (
        post_reference.loc[can_backfill_mask, "first_comment_timestamp"] - MIDNIGHT_BACKOFF
    )

    return post_reference[["media_id", "delay_post_timestamp"]].copy()


def build_window_metrics(
    window: WindowSpec,
    comments: pd.DataFrame,
    posts: pd.DataFrame,
    history: pd.DataFrame,
    first_comment_overall: pd.DataFrame,
    post_delay_reference: pd.DataFrame,
) -> pd.DataFrame:
    posts_mask = (posts["timestamp"] >= window.start) & (posts["timestamp"] < window.end_exclusive)
    comments_mask = (comments["timestamp"] >= window.start) & (comments["timestamp"] < window.end_exclusive)

    window_posts = posts.loc[posts_mask].copy()
    window_comments = comments.loc[comments_mask].copy()
    posts_in_window = int(len(window_posts))

    if posts_in_window == 0 or window_comments.empty:
        return pd.DataFrame(columns=TARGET_COLUMNS)

    window_comments = window_comments.sort_values(["user_key", "timestamp", "comment_id"], kind="mergesort").reset_index(drop=True)

    posts_seen = np.searchsorted(
        window_posts["timestamp"].to_numpy(dtype="datetime64[ns]"),
        window_comments["timestamp"].to_numpy(dtype="datetime64[ns]"),
        side="right",
    )
    window_comments["posts_seen_by_comment_time"] = posts_seen

    window_comments = window_comments.merge(post_delay_reference, on="media_id", how="left")
    window_comments["delay_hours"] = (
        (window_comments["timestamp"] - window_comments["delay_post_timestamp"]) / pd.Timedelta(hours=1)
    )
    window_comments["delay_hours"] = window_comments["delay_hours"].clip(lower=0)

    per_post_counts = (
        window_comments.groupby(["user_key", "media_id"], sort=False)
        .size()
        .rename("comments_on_post")
        .reset_index()
    )
    per_post_counts["comment_share"] = per_post_counts["comments_on_post"] / per_post_counts.groupby("user_key")["comments_on_post"].transform("sum")
    gini = (
        per_post_counts.assign(comment_share_sq=lambda df: np.square(df["comment_share"]))
        .groupby("user_key", sort=False)["comment_share_sq"]
        .sum()
        .rename("hhi")
        .reset_index()
    )
    gini["gini"] = 1 - gini["hhi"]

    latest_history = (
        history.loc[history["timestamp"] < window.end_exclusive, ["user_key", "timestamp"]]
        .groupby("user_key", sort=False)["timestamp"]
        .max()
        .rename("latest_history_comment")
        .reset_index()
    )

    aggregated = (
        window_comments.groupby("user_key", sort=False)
        .agg(
            user_id=("from_id", "last"),
            username=("from_username", "last"),
            comment_count=("comment_id", "size"),
            distinct_posts_commented=("media_id", "nunique"),
            last_comment_timestamp=("timestamp", "max"),
            last_posts_seen=("posts_seen_by_comment_time", "last"),
            engagement=("word_count", "mean"),
            delay=("delay_hours", "mean"),
        )
        .reset_index()
    )

    aggregated = aggregated.merge(gini[["user_key", "gini"]], on="user_key", how="left")
    aggregated = aggregated.merge(first_comment_overall, on="user_key", how="left")
    aggregated = aggregated.merge(latest_history, on="user_key", how="left")

    aggregated["recency"] = (posts_in_window - aggregated["last_posts_seen"]) / posts_in_window
    aggregated["coverage"] = aggregated["distinct_posts_commented"] / posts_in_window
    aggregated["tenure"] = (
        (aggregated["latest_history_comment"] - aggregated["first_comment_timestamp"]) / pd.Timedelta(days=1)
    )

    aggregated["window_id"] = window.window_id
    aggregated["window_label"] = window.window_label
    aggregated["window_index"] = window.window_index
    aggregated["window_start"] = window.start.date().isoformat()
    aggregated["window_end"] = window.end.date().isoformat()
    aggregated["window_total_days"] = window.total_days
    aggregated["posts_in_window"] = posts_in_window

    metrics = aggregated[TARGET_COLUMNS].copy()
    for column in ["recency", "coverage", "engagement", "tenure", "gini"]:
        metrics[column] = metrics[column].round(6)

    return metrics


first_comment_overall = (
    history_comments.groupby("user_key", sort=False)["timestamp"]
    .min()
    .rename("first_comment_timestamp")
    .reset_index()
)

post_delay_reference = build_post_delay_reference(base_posts, base_comments)
windows = build_window_specs()

metrics_frames = [
    build_window_metrics(
        window=window,
        comments=base_comments,
        posts=base_posts,
        history=history_comments,
        first_comment_overall=first_comment_overall,
        post_delay_reference=post_delay_reference,
    )
    for window in windows
]

metrics = pd.concat(metrics_frames, ignore_index=True)
metrics = metrics.sort_values(["window_index", "user_key"], kind="mergesort").reset_index(drop=True)
metrics.to_csv(METRICS_OUTPUT, index=False, encoding="utf-8-sig")

print(f"Saved {len(metrics):,} rows to {METRICS_OUTPUT.name}")
print(f"Rolling windows: {len(windows):,}")
print(f"First window: {windows[0].window_label}")
print(f"Last window: {windows[-1].window_label}")
metrics.head()


## Plot selected 2025 rolling windows

We compare the first five rolling quadrimesters of 2025 in a single 2x3 grid.

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt


PLOT_OUTPUT = OUTPUT_DIR / "ig_rolling_quadrimesters_2025_first5_distributions.png"
PLOT_VARIABLES = [
    ("recency", "Recency"),
    ("coverage", "Coverage"),
    ("engagement", "Engagement"),
    ("delay", "Delay"),
    ("tenure", "Tenure"),
    ("gini", "Gini"),
]
PLOT_WINDOW_STARTS = [
    "2025-01-01",
    "2025-02-01",
    "2025-03-01",
    "2025-04-01",
    "2025-05-01",
]
PLOT_COLORS = ["#0F4C5C", "#E36414", "#6A994E", "#8D5A97", "#BC4749"]
THRESHOLD_PERCENTILES = {
    "recency": (0.30, 0.70),
    "coverage": (0.60, 0.90),
    "delay": (0.685565, 0.874866),
    "tenure": (0.30, 0.70),
}
FIXED_THRESHOLDS = {
    "engagement": (5.0, 10.0),
    "gini": (0.20, 0.60),
}
BANDWIDTH_MULTIPLIERS = {
    "delay": 1.35,
}
THRESHOLD_STYLE = {
    "color": "#3A3A3A",
    "linestyle": (0, (4, 3)),
    "linewidth": 1.4,
    "alpha": 0.9,
}
PLOT_X_LIMITS = {
    "coverage": (0.0, 0.12),
    "delay": (0.0, 200.0),
}
BOUNDED_RANGES = {
    "recency": (0.0, 1.0),
    "coverage": (0.0, 1.0),
    "gini": (0.0, 1.0),
}


def estimate_bandwidth(values: np.ndarray, multiplier: float = 1.0) -> float:
    values = values[np.isfinite(values)]
    if values.size <= 1:
        baseline = abs(values[0]) if values.size == 1 else 1.0
        return max(baseline * 0.05 * multiplier, 1e-3)

    std = np.std(values, ddof=1)
    iqr = np.subtract(*np.percentile(values, [75, 25]))
    sigma = min(std, iqr / 1.34) if iqr > 0 else std
    if not np.isfinite(sigma) or sigma <= 0:
        sigma = max((values.max() - values.min()) / 6, 1e-3)
    return max(0.9 * sigma * values.size ** (-1 / 5) * multiplier, 1e-3)


def gaussian_kde_on_grid(values: np.ndarray, grid: np.ndarray, multiplier: float = 1.0) -> np.ndarray:
    values = values[np.isfinite(values)]
    if values.size == 0:
        return np.zeros_like(grid)

    bandwidth = estimate_bandwidth(values, multiplier=multiplier)
    scaled = (grid[:, None] - values[None, :]) / bandwidth
    kernel = np.exp(-0.5 * np.square(scaled)) / np.sqrt(2 * np.pi)
    return kernel.mean(axis=1) / bandwidth

plot_data = metrics.loc[metrics["window_start"].isin(PLOT_WINDOW_STARTS)].copy()
plot_windows = (
    plot_data[["window_start", "window_label", "window_index"]]
    .drop_duplicates()
    .sort_values("window_index")
    .reset_index(drop=True)
)
THRESHOLDS = {
    column: tuple(float(metrics[column].quantile(percentile)) for percentile in percentiles)
    for column, percentiles in THRESHOLD_PERCENTILES.items()
}
THRESHOLDS.update(FIXED_THRESHOLDS)

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.subplots_adjust(top=0.84, wspace=0.24, hspace=0.34)
axes = axes.ravel()

for axis, (column, label) in zip(axes, PLOT_VARIABLES):
    all_values = plot_data[column].dropna().to_numpy(dtype=float)
    x_min = float(all_values.min())
    x_max = float(all_values.max())
    if x_max > x_min:
        padding = 0.06 * (x_max - x_min)
    else:
        padding = max(abs(x_min) * 0.1, 1.0)

    lower = x_min - padding
    upper = x_max + padding
    if column in BOUNDED_RANGES:
        bound_low, bound_high = BOUNDED_RANGES[column]
        lower = max(lower, bound_low)
        upper = min(upper, bound_high)

    grid = np.linspace(lower, upper, 500)
    bandwidth_multiplier = BANDWIDTH_MULTIPLIERS.get(column, 1.0)

    for color, window in zip(PLOT_COLORS, plot_windows.itertuples(index=False)):
        window_values = plot_data.loc[plot_data["window_start"] == window.window_start, column].dropna().to_numpy(dtype=float)
        density = gaussian_kde_on_grid(window_values, grid, multiplier=bandwidth_multiplier)
        axis.plot(
            grid,
            density,
            linewidth=2.0,
            color=color,
            label=window.window_label,
        )

    for threshold in THRESHOLDS[column]:
        axis.axvline(threshold, **THRESHOLD_STYLE)

    axis.set_title(label)
    axis.set_xlabel(label)
    axis.set_ylabel("Density")
    axis.grid(alpha=0.2, linewidth=0.6)
    if column in PLOT_X_LIMITS:
        axis.set_xlim(*PLOT_X_LIMITS[column])

handles, labels = axes[0].get_legend_handles_labels()
fig.suptitle("Metric distributions across the first five 2025 rolling quadrimesters", fontsize=16, y=0.98)
fig.legend(handles, labels, loc="upper center", ncol=3, frameon=False, bbox_to_anchor=(0.5, 0.93))
fig.savefig(PLOT_OUTPUT, dpi=220, bbox_inches="tight")
plt.close(fig)

print(f"Saved plot to {PLOT_OUTPUT.name}")


## Classify user-window profiles

We assign Low, Mid, and High labels to each metric and export the RCEDTG classification table.

In [ ]:
CLASSIFICATION_SPECS = {
    "recency": {"low": THRESHOLDS["recency"][0], "high": THRESHOLDS["recency"][1], "low_op": "<=", "high_op": ">"},
    "coverage": {"low": THRESHOLDS["coverage"][0], "high": THRESHOLDS["coverage"][1], "low_op": "<=", "high_op": ">"},
    "engagement": {"low": THRESHOLDS["engagement"][0], "high": THRESHOLDS["engagement"][1], "low_op": "<", "high_op": ">="},
    "delay": {"low": THRESHOLDS["delay"][0], "high": THRESHOLDS["delay"][1], "low_op": "<=", "high_op": ">"},
    "tenure": {"low": THRESHOLDS["tenure"][0], "high": THRESHOLDS["tenure"][1], "low_op": "<=", "high_op": ">"},
    "gini": {"low": THRESHOLDS["gini"][0], "high": THRESHOLDS["gini"][1], "low_op": "<=", "high_op": ">"},
}


def build_low_mask(series: pd.Series, threshold: float, op: str) -> pd.Series:
    if op == "<=":
        return series <= threshold
    if op == "<":
        return series < threshold
    raise ValueError(f"Unsupported operator: {op}")


def build_high_mask(series: pd.Series, threshold: float, op: str) -> pd.Series:
    if op == ">":
        return series > threshold
    if op == ">=":
        return series >= threshold
    raise ValueError(f"Unsupported operator: {op}")


def classify_three_levels(series: pd.Series, low_threshold: float, high_threshold: float, low_op: str, high_op: str) -> pd.Series:
    low_mask = build_low_mask(series, low_threshold, low_op)
    high_mask = build_high_mask(series, high_threshold, high_op)
    levels = np.where(low_mask, "Low", np.where(high_mask, "High", "Mid"))
    return pd.Series(levels, index=series.index, dtype="string")


classified = metrics.copy()

for column, spec in CLASSIFICATION_SPECS.items():
    classified[f"{column}_low_threshold"] = round(float(spec["low"]), 6)
    classified[f"{column}_high_threshold"] = round(float(spec["high"]), 6)
    classified[f"{column}_level"] = classify_three_levels(
        classified[column],
        low_threshold=spec["low"],
        high_threshold=spec["high"],
        low_op=spec["low_op"],
        high_op=spec["high_op"],
    )

classified["rcedtg_profile"] = (
    "R=" + classified["recency_level"]
    + " | C=" + classified["coverage_level"]
    + " | E=" + classified["engagement_level"]
    + " | D=" + classified["delay_level"]
    + " | T=" + classified["tenure_level"]
    + " | G=" + classified["gini_level"]
)

CLASSIFICATION_COLUMNS = [
    "window_id",
    "window_label",
    "window_index",
    "window_start",
    "window_end",
    "window_total_days",
    "posts_in_window",
    "user_key",
    "user_id",
    "username",
    "comment_count",
    "last_comment_timestamp",
    "recency",
    "recency_low_threshold",
    "recency_high_threshold",
    "recency_level",
    "coverage",
    "coverage_low_threshold",
    "coverage_high_threshold",
    "coverage_level",
    "engagement",
    "engagement_low_threshold",
    "engagement_high_threshold",
    "engagement_level",
    "delay",
    "delay_low_threshold",
    "delay_high_threshold",
    "delay_level",
    "tenure",
    "tenure_low_threshold",
    "tenure_high_threshold",
    "tenure_level",
    "gini",
    "gini_low_threshold",
    "gini_high_threshold",
    "gini_level",
    "rcedtg_profile",
]

classified_output = classified[CLASSIFICATION_COLUMNS].copy()
classified_output.to_csv(CLASSIFICATION_OUTPUT, index=False, encoding="utf-8-sig")

print(f"Saved classification to {CLASSIFICATION_OUTPUT.name}")
classified_output.head()


## Build user frequency profiles

We aggregate the 40 rolling periods into one row per user by counting how often each metric is Low, Mid, or High, standardize the count features with Z-scores, and fit K-Means with k=4 as the final clustering solution.

In [ ]:
import os
os.environ.setdefault("LOKY_MAX_CPU_COUNT", "1")

from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler


PROFILE_VARIABLES = ["recency", "coverage", "engagement", "delay", "tenure", "gini"]
PROFILE_LEVELS = ["Low", "Mid", "High"]
FINAL_K = 4
KMEANS_RANDOM_SEED = 42
KMEANS_K4_VALUE_ORDER = {1: 1, 3: 2, 2: 3, 4: 4}
KMEANS_K4_NAME_MAP = {
    1: "Brand advocates",
    2: "Established regulars",
    3: "Passive regulars",
    4: "Occasional Visitors",
}

profile_source = classified_output.copy()

user_info = (
    profile_source
    .sort_values(["user_key", "window_index"])
    .groupby("user_key", as_index=True)
    .agg(
        user_id=("user_id", "first"),
        username=("username", "first"),
        observed_windows=("window_id", "nunique"),
    )
)

count_frames = []

for column in PROFILE_VARIABLES:
    level_column = f"{column}_level"
    counts = pd.crosstab(profile_source["user_key"], profile_source[level_column]).reindex(columns=PROFILE_LEVELS, fill_value=0)
    counts.columns = [f"{column}_{level.lower()}_count" for level in PROFILE_LEVELS]
    count_frames.append(counts)

user_profiles = pd.concat([user_info] + count_frames, axis=1).reset_index()
count_columns = [column for column in user_profiles.columns if column.endswith("_count")]

scaler = StandardScaler()
scaled_array = scaler.fit_transform(user_profiles[count_columns])
scaled_columns = [f"{column}_z" for column in count_columns]
scaled_frame = pd.DataFrame(scaled_array, columns=scaled_columns, index=user_profiles.index)
user_profiles = pd.concat([user_profiles, scaled_frame], axis=1)
user_profiles.to_csv(USER_PROFILE_OUTPUT, index=False, encoding="utf-8-sig")

scaled_values = user_profiles[scaled_columns].to_numpy(dtype=float)
kmeans_k4 = KMeans(n_clusters=FINAL_K, n_init=50, random_state=KMEANS_RANDOM_SEED)
kmeans_k4_raw_clusters = kmeans_k4.fit_predict(scaled_values) + 1
user_profiles["cluster"] = pd.Series(kmeans_k4_raw_clusters, index=user_profiles.index).map(KMEANS_K4_VALUE_ORDER).astype(int)
user_profiles["cluster_name"] = user_profiles["cluster"].map(KMEANS_K4_NAME_MAP)

summary_rows = []

for cluster_id, cluster_frame in user_profiles.groupby("cluster"):
    row = {
        "cluster": int(cluster_id),
        "cluster_name": KMEANS_K4_NAME_MAP[int(cluster_id)],
        "n_users": int(len(cluster_frame)),
        "avg_observed_windows": float(cluster_frame["observed_windows"].mean()),
    }

    for column in PROFILE_VARIABLES:
        low_total = float(cluster_frame[f"{column}_low_count"].sum())
        mid_total = float(cluster_frame[f"{column}_mid_count"].sum())
        high_total = float(cluster_frame[f"{column}_high_count"].sum())
        grand_total = low_total + mid_total + high_total

        low_share = low_total / grand_total if grand_total else np.nan
        mid_share = mid_total / grand_total if grand_total else np.nan
        high_share = high_total / grand_total if grand_total else np.nan

        row[f"{column}_low_share"] = low_share
        row[f"{column}_mid_share"] = mid_share
        row[f"{column}_high_share"] = high_share

        share_map = {"Low": low_share, "Mid": mid_share, "High": high_share}
        row[f"{column}_dominant_level"] = max(share_map, key=share_map.get)

    summary_rows.append(row)

cluster_summary = pd.DataFrame(summary_rows).sort_values("cluster").reset_index(drop=True)

assignment_columns = ["user_key", "user_id", "username", "observed_windows", "cluster", "cluster_name"] + count_columns + scaled_columns
user_profiles[assignment_columns].to_csv(KMEANS_K4_ASSIGNMENT_OUTPUT, index=False, encoding="utf-8-sig")
cluster_summary.to_csv(KMEANS_K4_SUMMARY_OUTPUT, index=False, encoding="utf-8-sig")

print(f"Saved user profiles to {USER_PROFILE_OUTPUT.name}")
print(f"Saved K-Means k=4 assignments to {KMEANS_K4_ASSIGNMENT_OUTPUT.name}")
display(cluster_summary)
user_profiles.head()


## Visualize K-Means clusters

We project the standardized user profiles into two PCA dimensions and show the K-Means partitions for k=3 and k=4 with their centroids.

In [ ]:
from sklearn.decomposition import PCA


VISUAL_K_VALUES = [3, 4]
VISUAL_CLUSTER_COLORS = [
    "#0F4C5C",
    "#E36414",
    "#6A994E",
    "#8D5A97",
    "#BC4749",
    "#3C6E71",
]
VISUAL_CLUSTER_PLOTS = {
    3: KMEANS_K3_PLOT,
    4: KMEANS_K4_PLOT,
}

if "user_profiles" not in globals():
    user_profiles = pd.read_csv(USER_PROFILE_OUTPUT, low_memory=False)

if "scaled_columns" not in globals():
    scaled_columns = [column for column in user_profiles.columns if column.endswith("_count_z")]

scaled_values = user_profiles[scaled_columns].to_numpy(dtype=float)
pca = PCA(n_components=2, random_state=KMEANS_RANDOM_SEED)
coordinates = pca.fit_transform(scaled_values)

for k in VISUAL_K_VALUES:
    model = KMeans(n_clusters=k, n_init=50, random_state=KMEANS_RANDOM_SEED)
    raw_labels = model.fit_predict(scaled_values) + 1
    centroid_coordinates = pca.transform(model.cluster_centers_)

    if k == 4:
        labels = pd.Series(raw_labels).map(KMEANS_K4_VALUE_ORDER).to_numpy(dtype=int)
        value_to_raw = {value: raw for raw, value in KMEANS_K4_VALUE_ORDER.items()}
        centroid_coordinates = np.vstack([centroid_coordinates[value_to_raw[cluster_id] - 1] for cluster_id in sorted(value_to_raw)])
    else:
        labels = raw_labels

    plot_frame = pd.DataFrame(
        {
            "pc1": coordinates[:, 0],
            "pc2": coordinates[:, 1],
            "cluster": labels,
        }
    )

    plt.figure(figsize=(8.5, 6.5))

    for cluster_id in sorted(plot_frame["cluster"].unique()):
        cluster_frame = plot_frame.loc[plot_frame["cluster"] == cluster_id]
        color = VISUAL_CLUSTER_COLORS[(cluster_id - 1) % len(VISUAL_CLUSTER_COLORS)]
        plt.scatter(
            cluster_frame["pc1"],
            cluster_frame["pc2"],
            s=14,
            c=color,
            alpha=0.42,
            edgecolors="none",
            label=f"Cluster {cluster_id}",
        )

    plt.scatter(
        centroid_coordinates[:, 0],
        centroid_coordinates[:, 1],
        s=180,
        c="black",
        marker="X",
        edgecolors="white",
        linewidths=1.0,
        label="Centroids",
    )

    for cluster_id, (x_coord, y_coord) in enumerate(centroid_coordinates, start=1):
        plt.text(x_coord + 0.04, y_coord + 0.04, f"C{cluster_id}", fontsize=9, weight="bold")

    plt.title(f"K-Means clusters on user frequency profiles (k={k})")
    plt.xlabel("PC1")
    plt.ylabel("PC2")
    plt.grid(alpha=0.18)
    plt.legend(loc="upper center", bbox_to_anchor=(0.5, -0.1), ncol=min(k + 1, 4), frameon=False)
    plt.tight_layout()
    plt.savefig(VISUAL_CLUSTER_PLOTS[k], dpi=220, bbox_inches="tight")
    plt.close()

print(f"Saved {KMEANS_K3_PLOT.name} and {KMEANS_K4_PLOT.name}")


## Build the monthly cluster matrix

We map each active user-window RCEDTG profile to the nearest K-Means k=4 cluster prototype in ordinal RCEDTG space, then build a user-by-month matrix from January 2023 onward with C1-C4, Inactive, and Not yet active.

In [ ]:
PROFILE_TO_CLUSTER_LEVEL_MAP = {"Low": 1, "Mid": 2, "High": 3}
PROFILE_VARIABLES = ["recency", "coverage", "engagement", "delay", "tenure", "gini"]
PROFILE_LEVEL_COLUMNS = [f"{column}_level" for column in PROFILE_VARIABLES]

classification_matrix_source = classified_output.copy()
cluster_summary_source = pd.read_csv(KMEANS_K4_SUMMARY_OUTPUT)
comments_source = pd.read_csv(COMMENTS_OUTPUT, usecols=["user_key", "from_id", "from_username", "timestamp"], low_memory=False)

cluster_prototypes = []

for _, row in cluster_summary_source.sort_values("cluster").iterrows():
    prototype = []
    for column in PROFILE_VARIABLES:
        expected_level = 1 * row[f"{column}_low_share"] + 2 * row[f"{column}_mid_share"] + 3 * row[f"{column}_high_share"]
        prototype.append(expected_level)
    cluster_prototypes.append(prototype)

cluster_prototypes = np.asarray(cluster_prototypes, dtype=float)

active_vectors = classification_matrix_source[PROFILE_LEVEL_COLUMNS].replace(PROFILE_TO_CLUSTER_LEVEL_MAP).to_numpy(dtype=float)
prototype_distances = np.abs(active_vectors[:, None, :] - cluster_prototypes[None, :, :]).sum(axis=2)
classification_matrix_source["dynamic_cluster"] = "C" + (prototype_distances.argmin(axis=1) + 1).astype(str)

window_lookup = (
    classification_matrix_source[["window_id", "window_label", "window_index", "window_start", "window_end"]]
    .drop_duplicates()
    .assign(window_start=lambda frame: pd.to_datetime(frame["window_start"]), window_end=lambda frame: pd.to_datetime(frame["window_end"]))
    .sort_values("window_index")
)
window_lookup = window_lookup.loc[window_lookup["window_end"].dt.to_period("M") >= pd.Period("2023-01", freq="M")].copy()
window_lookup["end_month"] = window_lookup["window_end"].dt.to_period("M").astype(str)

first_comments = (
    comments_source.assign(timestamp=pd.to_datetime(comments_source["timestamp"]))
    .sort_values(["user_key", "timestamp"])
    .groupby("user_key", as_index=False)
    .agg(
        user_id=("from_id", "first"),
        username=("from_username", "first"),
        first_comment_timestamp=("timestamp", "min"),
    )
)

user_base = (
    classification_matrix_source[["user_key", "user_id", "username"]]
    .sort_values(["user_key", "username"], na_position="last")
    .groupby("user_key", as_index=False)
    .agg(user_id=("user_id", "first"), username=("username", "first"))
    .merge(first_comments[["user_key", "first_comment_timestamp"]], on="user_key", how="left")
    .sort_values(["username", "user_key"], na_position="last")
    .reset_index(drop=True)
)

active_assignments = classification_matrix_source[["user_key", "window_id", "dynamic_cluster"]].copy()
monthly_matrix = user_base[["user_key", "user_id", "username"]].copy()

for _, window_row in window_lookup.iterrows():
    column_name = window_row["end_month"]
    window_id = window_row["window_id"]
    window_start = window_row["window_start"]

    merged = monthly_matrix[["user_key"]].merge(
        active_assignments.loc[active_assignments["window_id"] == window_id, ["user_key", "dynamic_cluster"]],
        on="user_key",
        how="left",
    )

    active_mask = merged["dynamic_cluster"].notna()
    first_comment = user_base["first_comment_timestamp"]
    not_yet_active_mask = first_comment.isna() | (first_comment >= window_start)

    monthly_matrix[column_name] = np.where(
        active_mask,
        merged["dynamic_cluster"],
        np.where(not_yet_active_mask, "Not yet active", "Inactive"),
    )

monthly_matrix.to_csv(MONTHLY_CLUSTER_MATRIX_OUTPUT, index=False, encoding="utf-8-sig")

print(f"Saved monthly cluster matrix to {MONTHLY_CLUSTER_MATRIX_OUTPUT.name}")
monthly_matrix.head()


## Plot 4-month transition matrices

We pool the consecutive monthly transitions inside each 4-month block from January 2023 onward, save one row-normalized transition matrix per block, and also build yearly side-by-side summaries.

In [ ]:
from matplotlib.colors import LinearSegmentedColormap

STATE_NAME_MAP = {
    "C1": "Brand advocates",
    "C2": "Established regulars",
    "C3": "Passive regulars",
    "C4": "Occasional Visitors",
    "Inactive": "Inactive",
    "Not yet active": "Not yet active",
}
STATE_ORDER = [
    "Brand advocates",
    "Established regulars",
    "Passive regulars",
    "Occasional Visitors",
    "Inactive",
    "Not yet active",
]
TRANSITION_CMAP = LinearSegmentedColormap.from_list(
    "transition_blue_red",
    ["#2166AC", "#67A9CF", "#D1E5F0", "#FDDBC7", "#EF8A62", "#B2182B"],
)

TRANSITION_MATRIX_DIR.mkdir(parents=True, exist_ok=True)
for old_plot in TRANSITION_MATRIX_DIR.glob("*.png"):
    old_plot.unlink()

legacy_transition_plot = OUTPUT_DIR / "Ig_RCEDTG_k4_transition_matrices_4month_blocks.png"
if legacy_transition_plot.exists():
    legacy_transition_plot.unlink()

monthly_cluster_matrix = pd.read_csv(MONTHLY_CLUSTER_MATRIX_OUTPUT, low_memory=False)
month_columns = monthly_cluster_matrix.columns[3:].tolist()

block_months = []
for start_index in range(0, len(month_columns), 4):
    block = month_columns[start_index:start_index + 4]
    if len(block) == 4:
        block_months.append(block)

saved_plots = []
block_probability_maps = {}

for block in block_months:
    start_month = block[0]
    end_month = block[-1]

    transition_frames = []
    for month_from, month_to in zip(block[:-1], block[1:]):
        step_transitions = (
            monthly_cluster_matrix[[month_from, month_to]]
            .rename(columns={month_from: "state_from", month_to: "state_to"})
            .replace(STATE_NAME_MAP)
        )
        transition_frames.append(step_transitions)

    transitions = pd.concat(transition_frames, ignore_index=True)

    counts = pd.crosstab(transitions["state_from"], transitions["state_to"]).reindex(
        index=STATE_ORDER,
        columns=STATE_ORDER,
        fill_value=0,
    )
    probabilities = counts.div(counts.sum(axis=1).replace(0, np.nan), axis=0).fillna(0.0)

    block_probability_maps[(start_month, end_month)] = probabilities.copy()

    fig, axis = plt.subplots(figsize=(11.5, 9.5))
    image = axis.imshow(probabilities.to_numpy(), cmap=TRANSITION_CMAP, vmin=0.0, vmax=1.0)

    for row_index in range(len(STATE_ORDER)):
        for column_index in range(len(STATE_ORDER)):
            value = probabilities.iat[row_index, column_index]
            label = f"{value:.0%}" if value >= 0.005 else ""
            axis.text(
                column_index,
                row_index,
                label,
                ha="center",
                va="center",
                fontsize=10,
                color="black",
            )

    start_period = pd.Period(start_month, freq="M")
    end_period = pd.Period(end_month, freq="M")
    axis.set_title(
        f"Transition matrix: {start_period.strftime('%b')}-{end_period.strftime('%b')} {end_period.year}",
        fontsize=15,
        pad=14,
    )
    axis.set_xticks(range(len(STATE_ORDER)))
    axis.set_xticklabels(STATE_ORDER, rotation=45, ha="right", fontsize=10)
    axis.set_yticks(range(len(STATE_ORDER)))
    axis.set_yticklabels(STATE_ORDER, fontsize=10)
    axis.set_xlabel("Next monthly state within block", fontsize=11)
    axis.set_ylabel("Current monthly state within block", fontsize=11)

    colorbar = fig.colorbar(image, ax=axis, fraction=0.046, pad=0.04)
    colorbar.set_label("Row-normalized transition share", fontsize=11)

    fig.tight_layout()

    output_name = f"transition_matrix_{start_month}_to_{end_month}.png"
    output_path = TRANSITION_MATRIX_DIR / output_name
    fig.savefig(output_path, dpi=220, bbox_inches="tight")
    plt.close(fig)
    saved_plots.append(output_path.name)

yearly_summary_plots = []
for year in [2023, 2024, 2025]:
    year_blocks = [block for block in block_months if pd.Period(block[0], freq="M").year == year]
    if not year_blocks:
        continue

    fig, axes = plt.subplots(1, len(year_blocks), figsize=(20, 7.8))
    if len(year_blocks) == 1:
        axes = [axes]

    image = None
    for axis, block in zip(axes, year_blocks):
        start_month = block[0]
        end_month = block[-1]
        probabilities = block_probability_maps[(start_month, end_month)]
        image = axis.imshow(probabilities.to_numpy(), cmap=TRANSITION_CMAP, vmin=0.0, vmax=1.0)

        for row_index in range(len(STATE_ORDER)):
            for column_index in range(len(STATE_ORDER)):
                value = probabilities.iat[row_index, column_index]
                label = f"{value:.0%}" if value >= 0.005 else ""
                axis.text(
                    column_index,
                    row_index,
                    label,
                    ha="center",
                    va="center",
                    fontsize=8,
                    color="black",
                )

        start_period = pd.Period(start_month, freq="M")
        end_period = pd.Period(end_month, freq="M")
        axis.set_title(f"{start_period.strftime('%b')}-{end_period.strftime('%b')} {year}", fontsize=13, pad=10)
        axis.set_xticks(range(len(STATE_ORDER)))
        axis.set_xticklabels(STATE_ORDER, rotation=45, ha="right", fontsize=8)
        axis.set_yticks(range(len(STATE_ORDER)))
        axis.set_yticklabels(STATE_ORDER, fontsize=8)
        axis.set_xlabel("Next monthly state", fontsize=10)
        axis.set_ylabel("Current monthly state", fontsize=10)

    fig.suptitle(f"Transition matrices across 4-month blocks - {year}", fontsize=16, y=0.97)
    colorbar = fig.colorbar(image, ax=axes, location="right", fraction=0.03, pad=0.08, shrink=0.92)
    colorbar.set_label("Row-normalized transition share", fontsize=10)
    fig.subplots_adjust(left=0.06, right=0.86, bottom=0.2, top=0.88, wspace=0.32)

    yearly_output = TRANSITION_MATRIX_DIR / f"transition_matrices_{year}.png"
    fig.savefig(yearly_output, dpi=220, bbox_inches="tight")
    plt.close(fig)
    yearly_summary_plots.append(yearly_output.name)

print(f"Saved {len(saved_plots)} transition matrices to {TRANSITION_MATRIX_DIR.name}")
print(f"Saved {len(yearly_summary_plots)} yearly summaries to {TRANSITION_MATRIX_DIR.name}")
yearly_summary_plots


## Render summary tables

We rebuild the state-persistence summary tables with the new 4-cluster definition and save the three PNG tables in a dedicated folder.

In [ ]:
from textwrap import fill

SUMMARY_HEADER_COLOR = "#1F5AA6"
SUMMARY_ROW_ALT_COLOR = "#F5F7FB"
SUMMARY_EDGE_COLOR = "#D7DCE2"
SUMMARY_TEXT_COLOR = "#1F1F1F"
NOT_YET_ACTIVE_LABEL = "Not yet active"
INACTIVE_LABEL = "Inactive"
NON_ACTIVE_LABELS = {NOT_YET_ACTIVE_LABEL, INACTIVE_LABEL}
ACTIVE_STATE_NAME_MAP = {
    "C1": "Brand advocates",
    "C2": "Established regulars",
    "C3": "Passive regulars",
    "C4": "Occasional Visitors",
}
STATE_NAME_MAP = {
    **ACTIVE_STATE_NAME_MAP,
    NOT_YET_ACTIVE_LABEL: NOT_YET_ACTIVE_LABEL,
    INACTIVE_LABEL: INACTIVE_LABEL,
}
STATE_ORDER = ["C1", "C2", "C3", "C4", NOT_YET_ACTIVE_LABEL, INACTIVE_LABEL]


def build_monthly_state_long(monthly_matrix: pd.DataFrame) -> pd.DataFrame:
    month_columns = monthly_matrix.columns[3:].tolist()
    month_index_map = {column: index for index, column in enumerate(month_columns, start=1)}

    long = monthly_matrix.melt(
        id_vars=["user_key", "user_id", "username"],
        value_vars=month_columns,
        var_name="window_id",
        value_name="cluster_id",
    )
    long["window_index"] = long["window_id"].map(month_index_map).astype(int)
    long["window_label"] = long["window_id"]
    periods = pd.PeriodIndex(long["window_id"], freq="M")
    long["window_start"] = periods.start_time
    long["window_end"] = periods.end_time
    return long.sort_values(["user_key", "window_index"], kind="mergesort").reset_index(drop=True)


def build_spells(full_long: pd.DataFrame) -> pd.DataFrame:
    ordered = full_long.sort_values(["user_key", "window_index"], kind="mergesort").reset_index(drop=True)
    ordered["prev_state"] = ordered.groupby("user_key", sort=False)["cluster_id"].shift(1)
    ordered["new_spell"] = (ordered["cluster_id"] != ordered["prev_state"]) | ordered["prev_state"].isna()
    ordered["spell_number"] = ordered.groupby("user_key", sort=False)["new_spell"].cumsum().astype(int)

    spells = (
        ordered.groupby(["user_key", "spell_number"], sort=False)
        .agg(
            user_id=("user_id", "last"),
            username=("username", "last"),
            state_id=("cluster_id", "last"),
            spell_start_window_index=("window_index", "min"),
            spell_end_window_index=("window_index", "max"),
            spell_start_window_id=("window_id", "first"),
            spell_end_window_id=("window_id", "last"),
            spell_start_window_label=("window_label", "first"),
            spell_end_window_label=("window_label", "last"),
            spell_start=("window_start", "first"),
            spell_end=("window_end", "last"),
            duration_updates=("window_index", "size"),
        )
        .reset_index()
    )

    spells = spells.sort_values(["user_key", "spell_number"], kind="mergesort").reset_index(drop=True)
    spells["next_state_id"] = spells.groupby("user_key", sort=False)["state_id"].shift(-1)
    spells["next_spell_start_window_index"] = spells.groupby("user_key", sort=False)["spell_start_window_index"].shift(-1)
    spells["next_spell_start_window_id"] = spells.groupby("user_key", sort=False)["spell_start_window_id"].shift(-1)
    spells["next_spell_start_window_label"] = spells.groupby("user_key", sort=False)["spell_start_window_label"].shift(-1)
    spells["is_censored"] = spells["next_state_id"].isna()
    spells["has_exit"] = ~spells["is_censored"]
    return spells


def compute_discrete_hazard(spells: pd.DataFrame, state_order: list[str]) -> pd.DataFrame:
    rows = []
    for state_id in state_order:
        state_spells = spells[spells["state_id"] == state_id].copy()
        if state_spells.empty:
            continue
        max_duration = int(state_spells["duration_updates"].max())
        for month_in_state in range(1, max_duration + 1):
            at_risk = int((state_spells["duration_updates"] >= month_in_state).sum())
            exits = int(((state_spells["duration_updates"] == month_in_state) & (state_spells["has_exit"])).sum())
            censored = int(((state_spells["duration_updates"] == month_in_state) & (state_spells["is_censored"])).sum())
            hazard = float(exits / at_risk) if at_risk > 0 else np.nan
            rows.append(
                {
                    "state_id": state_id,
                    "month_in_state": month_in_state,
                    "at_risk": at_risk,
                    "exits": exits,
                    "censored": censored,
                    "exit_hazard": hazard,
                }
            )
    return pd.DataFrame(rows)


def compute_tail_probabilities(hazard: pd.DataFrame, thresholds: list[int]) -> pd.DataFrame:
    rows = []
    for state_id, state_hazard in hazard.groupby("state_id", sort=False):
        state_hazard = state_hazard.sort_values("month_in_state", kind="mergesort").reset_index(drop=True)
        survival_ge = {1: 1.0}
        running = 1.0
        for _, row in state_hazard.iterrows():
            month_in_state = int(row["month_in_state"])
            running *= 1.0 - float(row["exit_hazard"])
            survival_ge[month_in_state + 1] = running

        row_out = {"state_id": state_id}
        for threshold in thresholds:
            row_out[f"prob_D_ge_{threshold}"] = float(survival_ge.get(threshold, 0.0))
        rows.append(row_out)
    return pd.DataFrame(rows)


def compute_duration_summary(spells: pd.DataFrame, hazard: pd.DataFrame, state_order: list[str]) -> pd.DataFrame:
    observed_summary = (
        spells.groupby("state_id", sort=False)
        .agg(
            spell_count=("state_id", "size"),
            completed_spell_count=("has_exit", "sum"),
            censored_spell_count=("is_censored", "sum"),
            mean_duration_observed=("duration_updates", "mean"),
            median_duration_observed=("duration_updates", "median"),
            max_duration_observed=("duration_updates", "max"),
        )
        .reset_index()
    )
    tail_probs = compute_tail_probabilities(hazard, thresholds=[2, 3, 4])
    summary = observed_summary.merge(tail_probs, on="state_id", how="left")
    summary["mean_duration_observed"] = summary["mean_duration_observed"].round(4)
    summary["median_duration_observed"] = summary["median_duration_observed"].round(4)
    state_rank = {state_id: rank for rank, state_id in enumerate(state_order, start=1)}
    summary["_rank"] = summary["state_id"].map(state_rank)
    summary = summary.sort_values("_rank", kind="mergesort").drop(columns="_rank").reset_index(drop=True)
    return summary


def compute_exit_destination_probabilities(spells: pd.DataFrame, state_order: list[str]) -> pd.DataFrame:
    exited = spells[spells["has_exit"]].copy()
    counts = exited.groupby(["state_id", "next_state_id"], sort=False).size().reset_index(name="exit_count")
    totals = counts.groupby("state_id", sort=False)["exit_count"].sum().rename("total_exits").reset_index()
    counts = counts.merge(totals, on="state_id", how="left")
    counts["exit_probability"] = counts["exit_count"] / counts["total_exits"]
    state_rank = {state_id: rank for rank, state_id in enumerate(state_order, start=1)}
    counts["_from_rank"] = counts["state_id"].map(state_rank)
    counts["_to_rank"] = counts["next_state_id"].map(state_rank)
    counts = counts.sort_values(["_from_rank", "_to_rank"], kind="mergesort").drop(columns=["_from_rank", "_to_rank"])
    return counts.reset_index(drop=True)


def format_probability(value: object) -> str:
    if pd.isna(value):
        return ""
    return f"{float(value):.3f}"


def format_duration(value: object) -> str:
    if pd.isna(value):
        return ""
    return f"{float(value):.2f}"


def classify_state_behavior(p_exit_within_1_month: float, p_exit_to_inactive_given_exit: float) -> str:
    if p_exit_to_inactive_given_exit >= 0.5:
        return "Pre-inactive"
    if p_exit_within_1_month <= 0.5:
        return "Stable"
    return "Transitional"


def build_duration_table(duration: pd.DataFrame) -> pd.DataFrame:
    table = duration[
        [
            "state_id",
            "state_name",
            "mean_duration_observed",
            "median_duration_observed",
            "prob_D_ge_2",
            "prob_D_ge_3",
            "prob_D_ge_4",
        ]
    ].copy()
    table = table.rename(
        columns={
            "state_id": "State",
            "state_name": "Name",
            "mean_duration_observed": "Mean D",
            "median_duration_observed": "Median D",
            "prob_D_ge_2": "P(D>=2)",
            "prob_D_ge_3": "P(D>=3)",
            "prob_D_ge_4": "P(D>=4)",
        }
    )
    for column in ["Mean D", "Median D"]:
        table[column] = table[column].map(format_duration)
    for column in ["P(D>=2)", "P(D>=3)", "P(D>=4)"]:
        table[column] = table[column].map(format_probability)
    return table


def build_exit_destination_table(exit_dest: pd.DataFrame, top_n: int = 3) -> pd.DataFrame:
    exit_dest = exit_dest.copy()
    exit_dest["rank"] = exit_dest.groupby("state_id", sort=False)["exit_probability"].rank(method="first", ascending=False)
    top = exit_dest[exit_dest["rank"] <= top_n].copy()
    top = top.sort_values(["state_id", "rank"], kind="mergesort")

    rows = []
    for state_id, state_rows in top.groupby("state_id", sort=False):
        row = {"State": str(state_id), "Name": str(state_rows["state_name"].iloc[0])}
        for _, state_row in state_rows.iterrows():
            rank = int(state_row["rank"])
            row[f"Top {rank}"] = str(state_row["next_state_id"])
            row[f"P{rank}"] = format_probability(state_row["exit_probability"])
        rows.append(row)

    table = pd.DataFrame(rows)
    expected_columns = ["State", "Name", "Top 1", "P1", "Top 2", "P2", "Top 3", "P3"]
    for column in expected_columns:
        if column not in table.columns:
            table[column] = ""
    state_rank = {state_id: rank for rank, state_id in enumerate(STATE_ORDER, start=1)}
    table["_rank"] = table["State"].map(state_rank)
    table = table.sort_values("_rank", kind="mergesort").drop(columns="_rank")
    return table[expected_columns].reset_index(drop=True)


def build_exit_risk_summary(duration: pd.DataFrame, exit_dest: pd.DataFrame) -> pd.DataFrame:
    active_duration = duration[~duration["state_id"].isin(NON_ACTIVE_LABELS)].copy()
    active_exit = exit_dest[~exit_dest["state_id"].isin(NON_ACTIVE_LABELS)].copy()

    p_exit_within_1 = active_duration[["state_id", "state_name", "mean_duration_observed", "prob_D_ge_2"]].copy()
    p_exit_within_1["p_exit_within_1_month"] = 1.0 - p_exit_within_1["prob_D_ge_2"]

    inactive_exit = (
        active_exit[active_exit["next_state_id"] == INACTIVE_LABEL][["state_id", "exit_probability"]]
        .rename(columns={"exit_probability": "p_exit_to_inactive_given_exit"})
        .copy()
    )

    active_exit_prob = (
        active_exit[~active_exit["next_state_id"].isin(NON_ACTIVE_LABELS)]
        .groupby("state_id", sort=False)["exit_probability"]
        .sum()
        .rename("p_exit_to_active_given_exit")
        .reset_index()
    )

    top_active_exit = (
        active_exit[~active_exit["next_state_id"].isin(NON_ACTIVE_LABELS)]
        .sort_values(["state_id", "exit_probability"], ascending=[True, False], kind="mergesort")
        .groupby("state_id", sort=False)
        .first()
        .reset_index()[["state_id", "next_state_id", "next_state_name", "exit_probability"]]
        .rename(
            columns={
                "next_state_id": "top_active_exit_state",
                "next_state_name": "top_active_exit_name",
                "exit_probability": "p_top_active_exit_given_exit",
            }
        )
    )

    summary = (
        p_exit_within_1.merge(inactive_exit, on="state_id", how="left")
        .merge(active_exit_prob, on="state_id", how="left")
        .merge(top_active_exit, on="state_id", how="left")
    )

    summary["p_exit_to_inactive_given_exit"] = summary["p_exit_to_inactive_given_exit"].fillna(0.0)
    summary["p_exit_to_active_given_exit"] = summary["p_exit_to_active_given_exit"].fillna(0.0)
    summary["p_top_active_exit_given_exit"] = summary["p_top_active_exit_given_exit"].fillna(0.0)
    summary = summary.rename(
        columns={
            "state_id": "State",
            "state_name": "Name",
            "mean_duration_observed": "Mean D",
            "p_exit_within_1_month": "P(exit<=1m)",
            "p_exit_to_inactive_given_exit": "P(to Inactive|exit)",
            "p_exit_to_active_given_exit": "P(to active|exit)",
            "top_active_exit_state": "Top active exit",
            "top_active_exit_name": "Top active exit name",
            "p_top_active_exit_given_exit": "P(top active|exit)",
        }
    )
    summary["Mean D"] = summary["Mean D"].round(4)
    summary["P(exit<=1m)"] = summary["P(exit<=1m)"].round(6)
    summary["P(to Inactive|exit)"] = summary["P(to Inactive|exit)"].round(6)
    summary["P(to active|exit)"] = summary["P(to active|exit)"].round(6)
    summary["P(top active|exit)"] = summary["P(top active|exit)"].round(6)
    summary["Behavioral role"] = summary.apply(
        lambda row: classify_state_behavior(float(row["P(exit<=1m)"]), float(row["P(to Inactive|exit)"])),
        axis=1,
    )
    state_rank = {state_id: rank for rank, state_id in enumerate(["C1", "C2", "C3", "C4"], start=1)}
    summary["_rank"] = summary["State"].map(state_rank)
    summary = summary.sort_values("_rank", kind="mergesort").drop(columns="_rank")
    return summary[
        [
            "State",
            "Name",
            "Behavioral role",
            "Mean D",
            "P(exit<=1m)",
            "P(to Inactive|exit)",
            "P(to active|exit)",
            "Top active exit",
            "Top active exit name",
            "P(top active|exit)",
        ]
    ].reset_index(drop=True)


def render_table_png(table_df: pd.DataFrame, title: str, output_path: Path, fig_width: float, fig_height: float, font_size: int, note_text: str) -> None:
    fig, ax = plt.subplots(figsize=(fig_width, fig_height))
    ax.axis("off")

    table = ax.table(cellText=table_df.values, colLabels=table_df.columns, cellLoc="center", loc="center")
    table.auto_set_font_size(False)
    table.set_fontsize(font_size)
    table.scale(1, 1.55)

    for (row, col), cell in table.get_celld().items():
        cell.set_edgecolor(SUMMARY_EDGE_COLOR)
        cell.set_linewidth(0.8)
        cell.get_text().set_color(SUMMARY_TEXT_COLOR)
        if row == 0:
            cell.set_facecolor(SUMMARY_HEADER_COLOR)
            cell.get_text().set_color("white")
            cell.get_text().set_weight("bold")
        else:
            cell.set_facecolor(SUMMARY_ROW_ALT_COLOR if row % 2 == 0 else "white")

    ax.set_title(title, fontsize=15, weight="bold", color=SUMMARY_TEXT_COLOR, pad=18)
    fig.text(0.02, 0.02, fill(note_text, width=135), ha="left", va="bottom", fontsize=max(font_size - 1, 9), color=SUMMARY_TEXT_COLOR)
    fig.tight_layout(rect=[0, 0.08, 1, 1])
    fig.savefig(output_path, dpi=220, bbox_inches="tight")
    plt.close(fig)


SUMMARY_TABLES_DIR.mkdir(parents=True, exist_ok=True)
for old_plot in SUMMARY_TABLES_DIR.glob("*.png"):
    old_plot.unlink()

monthly_cluster_matrix = pd.read_csv(MONTHLY_CLUSTER_MATRIX_OUTPUT, low_memory=False)
full_long = build_monthly_state_long(monthly_cluster_matrix)
spells = build_spells(full_long)
hazard = compute_discrete_hazard(spells, STATE_ORDER)
duration_summary = compute_duration_summary(spells, hazard, STATE_ORDER)
exit_destinations = compute_exit_destination_probabilities(spells, STATE_ORDER)

state_metadata = pd.DataFrame(
    {
        "state_id": list(STATE_NAME_MAP.keys()),
        "state_name": list(STATE_NAME_MAP.values()),
    }
)
duration_summary = duration_summary.merge(state_metadata, on="state_id", how="left")
exit_destinations = exit_destinations.merge(state_metadata, on="state_id", how="left")
next_state_metadata = state_metadata.rename(columns={"state_id": "next_state_id", "state_name": "next_state_name"})
exit_destinations = exit_destinations.merge(next_state_metadata, on="next_state_id", how="left")

duration_table = build_duration_table(duration_summary)
exit_destination_table = build_exit_destination_table(exit_destinations, top_n=3)
exit_risk_summary = build_exit_risk_summary(duration_summary, exit_destinations)

exit_risk_display = exit_risk_summary.copy()
exit_risk_display["Mean D"] = exit_risk_display["Mean D"].map(lambda value: f"{float(value):.2f}")
for column in ["P(exit<=1m)", "P(to Inactive|exit)", "P(to active|exit)", "P(top active|exit)"]:
    exit_risk_display[column] = exit_risk_display[column].map(format_probability)
exit_risk_display = exit_risk_display[
    [
        "State",
        "Name",
        "Behavioral role",
        "Mean D",
        "P(exit<=1m)",
        "P(to Inactive|exit)",
        "P(to active|exit)",
        "Top active exit",
        "P(top active|exit)",
    ]
]

render_table_png(
    duration_table,
    "State Persistence Summary - New 4 Clusters",
    STATE_PERSISTENCE_SUMMARY_PNG,
    fig_width=13.2,
    fig_height=5.2,
    font_size=11,
    note_text=(
        "D is the spell duration measured in consecutive monthly updates. "
        "P(D>=m) is the probability that a spell in state c lasts at least m updates, "
        "estimated from the discrete survival implied by the exit hazard."
    ),
)
render_table_png(
    exit_destination_table,
    "Top Exit Destinations by State - New 4 Clusters",
    STATE_EXIT_DESTINATIONS_PNG,
    fig_width=12.8,
    fig_height=5.2,
    font_size=11,
    note_text=(
        "P(exit to b | exit from c) is computed as the share of completed spells leaving state c "
        "whose next observed state is b."
    ),
)
render_table_png(
    exit_risk_display,
    "One-Month Exit Risk and Exit Direction by Active State - New 4 Clusters",
    STATE_EXIT_RISK_SUMMARY_PNG,
    fig_width=18.0,
    fig_height=5.6,
    font_size=10,
    note_text=(
        "P(exit<=1m) = 1 - P(D>=2), i.e. the probability that a spell in state c ends by the next monthly update. "
        "P(to Inactive|exit) is the share of completed spells leaving c whose next observed state is Inactive. "
        "The pre-activation state Not yet active is tracked separately and is excluded from this active-state summary. "
        "P(to active|exit) aggregates all destinations belonging to the four active clusters."
    ),
)

print(f"Saved {STATE_PERSISTENCE_SUMMARY_PNG.name}")
print(f"Saved {STATE_EXIT_DESTINATIONS_PNG.name}")
print(f"Saved {STATE_EXIT_RISK_SUMMARY_PNG.name}")


## Render the checkpoint Sankey diagram

We cluster active user-window observations directly in ordinal RCEDTG space, rebuild the monthly state matrix from those dynamic clusters, and render the Jan/May/Sep checkpoint Sankey diagram.

In [ ]:
from dataclasses import dataclass
import matplotlib.patches as patches
from matplotlib.path import Path as MplPath

SUNKEY_STAGE_SPECS = [
    ("Jan 2023", "2023-01", "Rolling window: Oct 2022-Jan 2023"),
    ("May 2023", "2023-05", "Rolling window: Feb-May 2023"),
    ("Sep 2023", "2023-09", "Rolling window: Jun-Sep 2023"),
    ("Jan 2024", "2024-01", "Rolling window: Oct 2023-Jan 2024"),
    ("May 2024", "2024-05", "Rolling window: Feb-May 2024"),
    ("Sep 2024", "2024-09", "Rolling window: Jun-Sep 2024"),
    ("Jan 2025", "2025-01", "Rolling window: Oct 2024-Jan 2025"),
    ("May 2025", "2025-05", "Rolling window: Feb-May 2025"),
    ("Sep 2025", "2025-09", "Rolling window: Jun-Sep 2025"),
    ("Jan 2026", "2026-01", "Rolling window: Oct 2025-Jan 2026"),
]
WINDOW_LEVEL_CLUSTER_NAMES = {
    1: "Brand advocates",
    2: "Expressive regulars",
    3: "Passive regulars",
    4: "Delayed visitors",
}
WINDOW_LEVEL_STATE_ORDER = ["C1", "C2", "C3", "C4", "Not yet active", "Inactive"]
WINDOW_LEVEL_STATE_NAMES = {
    "C1": "Brand advocates",
    "C2": "Expressive regulars",
    "C3": "Passive regulars",
    "C4": "Delayed visitors",
    "Not yet active": "Not yet active",
    "Inactive": "Inactive",
}
WINDOW_LEVEL_STATE_COLORS = {
    "C1": "#0B6E4F",
    "C2": "#D17B0F",
    "C3": "#4E79A7",
    "C4": "#8E5EA2",
    "Not yet active": "#D8C9A7",
    "Inactive": "#D9D9D9",
}


@dataclass(frozen=True)
class SankeyNodeLayout:
    stage_name: str
    state: str
    count: int
    x: float
    y0: float
    y1: float


def sankey_with_alpha(hex_color: str, alpha: float) -> tuple[float, float, float, float]:
    hex_color = hex_color.lstrip("#")
    rgb = tuple(int(hex_color[index:index + 2], 16) / 255 for index in (0, 2, 4))
    return (*rgb, alpha)


def build_sankey_layout(stage_counts: dict[str, pd.Series], stage_names: list[str], state_order: list[str]) -> tuple[dict[tuple[str, str], SankeyNodeLayout], float]:
    total_users = int(stage_counts[stage_names[0]].sum())
    top_margin = 0.16
    bottom_margin = 0.07
    gap = 0.009
    usable_height = 1.0 - top_margin - bottom_margin - gap * (len(state_order) - 1)
    scale = usable_height / total_users
    x_positions = np.linspace(0.055, 0.88, len(stage_names))
    layout = {}

    for x, stage_name in zip(x_positions, stage_names):
        y = 1.0 - top_margin
        for state in state_order:
            count = int(stage_counts[stage_name].get(state, 0))
            height = count * scale
            y0 = y - height
            layout[(stage_name, state)] = SankeyNodeLayout(stage_name=stage_name, state=state, count=count, x=float(x), y0=float(y0), y1=float(y))
            y = y0 - gap

    return layout, scale


def draw_sankey_flow(ax: plt.Axes, left_node: SankeyNodeLayout, right_node: SankeyNodeLayout, left_offset: float, right_offset: float, height: float, color: str, node_width: float) -> None:
    x0 = left_node.x + node_width
    x1 = right_node.x
    y0_top = left_node.y1 - left_offset
    y0_bottom = y0_top - height
    y1_top = right_node.y1 - right_offset
    y1_bottom = y1_top - height
    ctrl_dx = (x1 - x0) * 0.36

    vertices = [
        (x0, y0_top),
        (x0 + ctrl_dx, y0_top),
        (x1 - ctrl_dx, y1_top),
        (x1, y1_top),
        (x1, y1_bottom),
        (x1 - ctrl_dx, y1_bottom),
        (x0 + ctrl_dx, y0_bottom),
        (x0, y0_bottom),
        (x0, y0_top),
    ]
    codes = [
        MplPath.MOVETO,
        MplPath.CURVE4,
        MplPath.CURVE4,
        MplPath.CURVE4,
        MplPath.LINETO,
        MplPath.CURVE4,
        MplPath.CURVE4,
        MplPath.CURVE4,
        MplPath.CLOSEPOLY,
    ]

    patch = patches.PathPatch(
        MplPath(vertices, codes),
        facecolor=sankey_with_alpha(color, 0.40),
        edgecolor=sankey_with_alpha(color, 0.54),
        linewidth=0.30,
        zorder=1,
    )
    ax.add_patch(patch)


def format_sankey_state_label(state: str) -> str:
    if state in {"Not yet active", "Inactive"}:
        return WINDOW_LEVEL_STATE_NAMES[state]
    return f"{state} - {WINDOW_LEVEL_STATE_NAMES[state]}"


def filter_always_inactive_users(df: pd.DataFrame, stage_names: list[str]) -> tuple[pd.DataFrame, int]:
    always_inactive_mask = df[stage_names].eq("Inactive").all(axis=1)
    removed_users = int(always_inactive_mask.sum())
    filtered = df.loc[~always_inactive_mask].copy()
    return filtered.reset_index(drop=True), removed_users


PROFILE_LEVEL_COLUMNS = ["recency_level", "coverage_level", "engagement_level", "delay_level", "tenure_level", "gini_level"]
LEVEL_TO_ORDINAL = {"Low": 1, "Mid": 2, "High": 3}
WINDOW_LEVEL_CLUSTER_VALUE_ORDER = {3: 1, 2: 2, 1: 3, 4: 4}

window_level_source = classified_output.copy()
window_level_vectors = window_level_source[PROFILE_LEVEL_COLUMNS].replace(LEVEL_TO_ORDINAL).to_numpy(dtype=float)
window_level_kmeans = KMeans(n_clusters=4, n_init=50, random_state=KMEANS_RANDOM_SEED)
window_level_raw_clusters = window_level_kmeans.fit_predict(window_level_vectors) + 1
window_level_source["dynamic_cluster_id"] = pd.Series(window_level_raw_clusters, index=window_level_source.index).map(WINDOW_LEVEL_CLUSTER_VALUE_ORDER).astype(int)
window_level_source["dynamic_cluster"] = "C" + window_level_source["dynamic_cluster_id"].astype(str)
window_level_source["dynamic_cluster_name"] = window_level_source["dynamic_cluster_id"].map(WINDOW_LEVEL_CLUSTER_NAMES)

window_level_summary_rows = []
for cluster_id, cluster_frame in window_level_source.groupby("dynamic_cluster_id"):
    row = {
        "cluster": int(cluster_id),
        "cluster_name": WINDOW_LEVEL_CLUSTER_NAMES[int(cluster_id)],
        "n_observations": int(len(cluster_frame)),
    }
    for column in ["recency", "coverage", "engagement", "delay", "tenure", "gini"]:
        level_column = f"{column}_level"
        level_shares = cluster_frame[level_column].value_counts(normalize=True)
        row[f"{column}_low_share"] = float(level_shares.get("Low", 0.0))
        row[f"{column}_mid_share"] = float(level_shares.get("Mid", 0.0))
        row[f"{column}_high_share"] = float(level_shares.get("High", 0.0))
        row[f"{column}_dominant_level"] = level_shares.idxmax()
    window_level_summary_rows.append(row)

window_level_cluster_summary = pd.DataFrame(window_level_summary_rows).sort_values("cluster").reset_index(drop=True)
window_level_cluster_summary.to_csv(WINDOWLEVEL_K4_SUMMARY_OUTPUT, index=False, encoding="utf-8-sig")

window_lookup = (
    window_level_source[["window_id", "window_label", "window_index", "window_start", "window_end"]]
    .drop_duplicates()
    .assign(window_start=lambda frame: pd.to_datetime(frame["window_start"]), window_end=lambda frame: pd.to_datetime(frame["window_end"]))
    .sort_values("window_index")
)
window_lookup = window_lookup.loc[window_lookup["window_end"].dt.to_period("M") >= pd.Period("2023-01", freq="M")].copy()
window_lookup["end_month"] = window_lookup["window_end"].dt.to_period("M").astype(str)

comments_source = pd.read_csv(COMMENTS_OUTPUT, usecols=["user_key", "from_id", "from_username", "timestamp"], low_memory=False)
first_comments = (
    comments_source.assign(timestamp=pd.to_datetime(comments_source["timestamp"]))
    .sort_values(["user_key", "timestamp"])
    .groupby("user_key", as_index=False)
    .agg(
        user_id=("from_id", "first"),
        username=("from_username", "first"),
        first_comment_timestamp=("timestamp", "min"),
    )
)

user_base = (
    window_level_source[["user_key", "user_id", "username"]]
    .sort_values(["user_key", "username"], na_position="last")
    .groupby("user_key", as_index=False)
    .agg(user_id=("user_id", "first"), username=("username", "first"))
    .merge(first_comments[["user_key", "first_comment_timestamp"]], on="user_key", how="left")
    .sort_values(["username", "user_key"], na_position="last")
    .reset_index(drop=True)
)

active_assignments = window_level_source[["user_key", "window_id", "dynamic_cluster"]].copy()
window_level_monthly_matrix = user_base[["user_key", "user_id", "username"]].copy()

for _, window_row in window_lookup.iterrows():
    column_name = window_row["end_month"]
    window_id = window_row["window_id"]
    window_start = window_row["window_start"]

    merged = window_level_monthly_matrix[["user_key"]].merge(
        active_assignments.loc[active_assignments["window_id"] == window_id, ["user_key", "dynamic_cluster"]],
        on="user_key",
        how="left",
    )

    active_mask = merged["dynamic_cluster"].notna()
    first_comment = user_base["first_comment_timestamp"]
    not_yet_active_mask = first_comment.isna() | (first_comment >= window_start)

    window_level_monthly_matrix[column_name] = np.where(
        active_mask,
        merged["dynamic_cluster"],
        np.where(not_yet_active_mask, "Not yet active", "Inactive"),
    )

window_level_monthly_matrix.to_csv(WINDOWLEVEL_K4_MONTHLY_MATRIX_OUTPUT, index=False, encoding="utf-8-sig")

SUNKEY_DIAGRAM_DIR.mkdir(parents=True, exist_ok=True)
for old_plot in SUNKEY_DIAGRAM_DIR.glob("*.png"):
    old_plot.unlink()

stage_names = [stage_name for stage_name, _, _ in SUNKEY_STAGE_SPECS]
input_columns = ["user_key", "user_id", "username"] + [column for _, column, _ in SUNKEY_STAGE_SPECS]
sankey_source = window_level_monthly_matrix[input_columns].rename(columns={column: stage_name for stage_name, column, _ in SUNKEY_STAGE_SPECS})
sankey_source, removed_users = filter_always_inactive_users(sankey_source, stage_names)

stage_counts = {
    stage_name: sankey_source[stage_name].value_counts().reindex(WINDOW_LEVEL_STATE_ORDER, fill_value=0)
    for stage_name in stage_names
}
state_rank = {state: index for index, state in enumerate(WINDOW_LEVEL_STATE_ORDER)}
links = []

for left_stage, right_stage in zip(stage_names[:-1], stage_names[1:]):
    grouped = sankey_source.groupby([left_stage, right_stage], dropna=False).size().reset_index(name="count")
    grouped = grouped.rename(columns={left_stage: "source_state", right_stage: "target_state"})
    grouped["source_stage"] = left_stage
    grouped["target_stage"] = right_stage
    grouped["source_state_order"] = grouped["source_state"].map(state_rank)
    grouped["target_state_order"] = grouped["target_state"].map(state_rank)
    grouped = grouped.sort_values(["source_state_order", "target_state_order"], kind="mergesort")
    for _, row in grouped.iterrows():
        links.append(
            {
                "source_stage": left_stage,
                "source_state": row["source_state"],
                "target_stage": right_stage,
                "target_state": row["target_state"],
                "count": int(row["count"]),
            }
        )

links_df = pd.DataFrame(links)
layout, scale = build_sankey_layout(stage_counts, stage_names, WINDOW_LEVEL_STATE_ORDER)
fig, ax = plt.subplots(figsize=(36, 12.5), dpi=220)
ax.set_xlim(0, 1.04)
ax.set_ylim(0, 1)
ax.axis("off")
fig.patch.set_facecolor("#F8F6F1")
ax.set_facecolor("#F8F6F1")

node_width = 0.018
source_offsets = {(stage_name, state): 0.0 for stage_name in stage_names for state in WINDOW_LEVEL_STATE_ORDER}
target_offsets = {(stage_name, state): 0.0 for stage_name in stage_names for state in WINDOW_LEVEL_STATE_ORDER}

for left_stage, right_stage in zip(stage_names[:-1], stage_names[1:]):
    pair_links = links_df[(links_df["source_stage"] == left_stage) & (links_df["target_stage"] == right_stage)].copy()
    pair_links["source_state_order"] = pair_links["source_state"].map(state_rank)
    pair_links["target_state_order"] = pair_links["target_state"].map(state_rank)
    pair_links = pair_links.sort_values(["source_state_order", "target_state_order"], kind="mergesort")

    for _, row in pair_links.iterrows():
        source_key = (left_stage, row["source_state"])
        target_key = (right_stage, row["target_state"])
        flow_height = int(row["count"]) * scale
        draw_sankey_flow(
            ax=ax,
            left_node=layout[source_key],
            right_node=layout[target_key],
            left_offset=source_offsets[source_key],
            right_offset=target_offsets[target_key],
            height=flow_height,
            color=WINDOW_LEVEL_STATE_COLORS.get(row["source_state"], "#999999"),
            node_width=node_width,
        )
        source_offsets[source_key] += flow_height
        target_offsets[target_key] += flow_height

for stage_name, _, stage_subtitle in SUNKEY_STAGE_SPECS:
    stage_total = int(stage_counts[stage_name].sum())
    stage_x = layout[(stage_name, WINDOW_LEVEL_STATE_ORDER[0])].x + node_width / 2
    ax.text(stage_x, 0.962, stage_name, ha="center", va="bottom", fontsize=12.6, fontweight="bold", color="#1F2A33")
    ax.text(stage_x, 0.944, stage_subtitle, ha="center", va="top", fontsize=8.2, color="#4A5A67")
    ax.text(stage_x, 0.926, f"Users: {stage_total:,}", ha="center", va="top", fontsize=8.2, color="#4A5A67")

for stage_name in stage_names:
    for state in WINDOW_LEVEL_STATE_ORDER:
        node = layout[(stage_name, state)]
        rect = patches.Rectangle((node.x, node.y0), node_width, node.y1 - node.y0, facecolor=WINDOW_LEVEL_STATE_COLORS.get(state, "#999999"), edgecolor="#FFFFFF", linewidth=0.8, zorder=2)
        ax.add_patch(rect)

        label_x = node.x + node_width + 0.0032
        label_y = (node.y0 + node.y1) / 2
        font_size = 6.7 if state not in {"Not yet active", "Inactive"} else 7.2
        if node.count * scale < 0.010:
            label_y = max(label_y, node.y0 + 0.0045)

        ax.text(label_x, label_y, f"{state} ({node.count:,})", ha="left", va="center", fontsize=font_size, color="#39434D", zorder=3)

fig.text(0.035, 0.978, "Sankey diagram of user state flows from January 2023 to January 2026 - Window-Level RCEDTG Clusters", ha="left", va="top", fontsize=19, fontweight="bold", color="#13212E")
fig.text(0.035, 0.953, "Active user-window observations are clustered directly in ordinal RCEDTG space. Checkpoints use the rolling quadrimesters ending in Jan 2023, May 2023, Sep 2023, Jan 2024, May 2024, Sep 2024, Jan 2025, May 2025, Sep 2025, and Jan 2026.", ha="left", va="top", fontsize=10.3, color="#405060")
fig.text(0.035, 0.932, f"{removed_users:,} users who stayed Inactive at every checkpoint were excluded. Not yet active marks users before their first active window, while Inactive marks users who had already activated at least once.", ha="left", va="top", fontsize=10.3, color="#405060")

legend_items = [state for state in WINDOW_LEVEL_STATE_ORDER if state not in {"Not yet active", "Inactive"}] + ["Not yet active", "Inactive"]
legend_y = 0.035
step = 0.125
legend_total_width = step * (len(legend_items) - 1) + 0.08
legend_x = 0.5 - legend_total_width / 2
for index, state in enumerate(legend_items):
    x = legend_x + index * step
    color = WINDOW_LEVEL_STATE_COLORS.get(state, "#999999")
    label = format_sankey_state_label(state)
    fig.patches.append(patches.Rectangle((x, legend_y - 0.008), 0.012, 0.012, transform=fig.transFigure, facecolor=color, edgecolor="none"))
    fig.text(x + 0.016, legend_y, label, ha="left", va="center", fontsize=9.0, color="#33414C")

plt.savefig(SUNKEY_DIAGRAM_PNG, bbox_inches="tight", facecolor=fig.get_facecolor())
plt.close(fig)

print(f"Saved {WINDOWLEVEL_K4_SUMMARY_OUTPUT.name}")
print(f"Saved {WINDOWLEVEL_K4_MONTHLY_MATRIX_OUTPUT.name}")
print(f"Saved {SUNKEY_DIAGRAM_PNG.name}")
print(f"Excluded always-inactive users: {removed_users:,}")
print(f"Users kept in diagram: {len(sankey_source):,}")
